# Phase 22 / Notebook 01 -- Setup, VRAM scaling, Phase 1 pretraining

**Goal:** train a 183,830,791-parameter byte-level foundation on REAL Hugging Face
corpora (raw next-byte causal LM, no prompts, no ReAct), on a Colab T4.

Pipeline: `01` (this notebook) -> `02_distill_and_align.ipynb` -> `03_rl_and_agent.ipynb`.
Checkpoints: `checkpoint_phase1.pt`, `checkpoint_phase2.pt`, `final_model.pt`
(saved locally AND to the Hub when `HF_REPO` is set).

**T4 precision warning:** `torch.bfloat16` CUDA autocast is **not** supported on
Turing/sm_75. `precision: auto` resolves to bfloat16 on Ampere+ and to float16 +
`GradScaler` on T4. The analytic budget shows ~3.97 GB at B=4/L=1024 with full
gradient checkpointing, so a measured fill ladder grows token throughput toward
the 10-12 GB target instead of faking utilisation.

**Resume:** every cell is safe to re-run. `Phase22Trainer` restores weights,
AdamW, GradScaler, RNG and the lossless byte-packer carry; Hub checkpoints are
downloaded automatically if the Colab VM was recycled.

In [ ]:
import json
import os
import pathlib
import subprocess
import sys

REPO = pathlib.Path(os.environ.get("AIRESEARCH_REPO", "/content/airesearch"))
if not (REPO / ".git").exists():
    subprocess.check_call(["git", "clone", "https://github.com/gabrielbosse1/airesearch.git", str(REPO)])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                       "datasets>=3,<5", "huggingface_hub>=0.25,<2", "pyyaml", "tqdm"])
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
print("repo", REPO)

In [ ]:
import torch

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type == "cuda":
    print("GPU", torch.cuda.get_device_name(DEVICE),
          "capability", torch.cuda.get_device_capability(DEVICE),
          "bf16_supported", torch.cuda.is_bf16_supported())
else:
    print("WARNING: no CUDA device. This notebook must run on a Colab T4 GPU for real training.")

## Architecture and VRAM budget

In [ ]:
from models.foundation import CausalByteTransformer
from training.phase22_common import (estimate_vram, load_config, model_config,
                                     resolve_precision, vram_fill_ladder, write_report)

CONFIG_PATH = "configs/t4_scaled.yaml"
cfg = load_config(CONFIG_PATH)
with torch.device("meta"):
    PARAMETERS = CausalByteTransformer(model_config(cfg)).parameters_count
print("parameters", PARAMETERS, "(target 150-200M)")
assert 150_000_000 <= PARAMETERS <= 200_000_000
est = estimate_vram(cfg, batch_size=4, seq_len=1024)
print("analytic VRAM estimate (GB)", est["estimated_total_gb"])
print("formula", est["formula"])

In [ ]:
plan = resolve_precision(DEVICE, cfg["training"]["requested_precision"],
                         cfg["training"]["precision"])
print(json.dumps(plan.as_dict(), indent=2))
# T4 = Turing sm_75: CUDA bf16 is NOT supported. `auto` falls back to fp16 +
# GradScaler. Both are 2-byte formats, so the VRAM budget is unchanged.
model = CausalByteTransformer(model_config(cfg)).to(DEVICE)
ladder = vram_fill_ladder(cfg, device=DEVICE, model=model, plan=plan)
print(json.dumps({"target_band_gb": ladder["target_band_gb"],
                  "default_estimate_gb": ladder["default_estimate_gb"],
                  "selected": ladder["selected"],
                  "target_met": ladder["target_met"],
                  "measured": ladder["measured"]}, indent=2))
write_report("runs/phase22/vram_fill_ladder.json", ladder)
if not ladder["measured"]:
    print("VRAM is UNMEASURED here; Colab will probe every candidate.")
print(json.dumps([{k: t[k] for k in ("micro_batch_size", "seq_len",
                                     "gradient_checkpointing", "estimated_gb",
                                     "peak_allocated_gb")} for t in ladder["trials"]], indent=2))
# Free the probe model before Phase22Trainer allocates its own copy.
del model
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()

## Hugging Face access + dataset discovery

In [ ]:
from huggingface_hub import login

# Set these in Colab secrets / environment. The checkpoint repo is REQUIRED for
# robust resume when the Colab VM is recycled.
HF_REPO = os.environ.get("HF_REPO", "")
HF_TOKEN = os.environ.get("HF_TOKEN", "")
if HF_TOKEN:
    login(token=HF_TOKEN)
cfg["checkpoints"]["hf_repo"] = HF_REPO or None
cfg["checkpoints"]["resume_from_hf"] = bool(HF_REPO)
print("HF checkpoint repo", cfg["checkpoints"]["hf_repo"] or "(not set - local resume only)")

In [ ]:
from data.phase22_sources import (HF_SEARCH_QUERIES, SourceRegistry, discover,
                                  stream_phase1)
from training.phase22_common import write_report

findings = {}
for query in HF_SEARCH_QUERIES[:8]:
    try:
        findings[query] = discover(query, limit=5)
    except Exception as exc:
        findings[query] = {"error": f"{type(exc).__name__}: {exc}"}
write_report("runs/phase22/hf_discovery.json", findings)
print("searched", len(findings), "queries")

registry = SourceRegistry()
sample = []
for source_id, raw in stream_phase1(cfg, limit_rows_per_source=2, registry=registry):
    sample.append({"source": source_id, "bytes": len(raw),
                   "preview": raw[:160].decode("utf-8", errors="replace")})
    if len(sample) >= 12:
        break
manifest = registry.manifest(cfg["data"]["license_manifest"],
                             extra={"phase": 1, "note": "streaming probe"})
print(json.dumps(sample, indent=2, ensure_ascii=False))
print(json.dumps({"sources": [r.as_dict() for r in registry.records]}, indent=2))

## Phase 1: raw next-byte causal LM

In [ ]:
from training.phase22_train import Phase22Trainer

MAX_STEPS = int(os.environ.get("PHASE1_MAX_STEPS", "0")) or None
trainer = Phase22Trainer(cfg, phase=1, device=DEVICE, data_mode="hub",
                         max_steps=MAX_STEPS)
report = trainer.run()
print(json.dumps({"steps": report["steps_completed"],
                  "resumed": report["resumed"],
                  "precision": report["precision"],
                  "parameters": report["parameters"],
                  "vram_measured": report["vram_measured"],
                  "checkpoint": report["checkpoint"]["path"],
                  "hf_upload": report["checkpoint"]["hf"],
                  "last_loss": report["last_loss"]}, indent=2))

In [ ]:
from training.phase22_common import hf_upload, save_deploy_copy

ck = cfg["checkpoints"]
path = trainer._checkpoint_path()
assert path.is_file(), f"missing {path}"
print("checkpoint_phase1.pt bytes", path.stat().st_size)
deploy = save_deploy_copy(path.with_name("checkpoint_phase1_fp16.pt"), path)
print("deploy copy", deploy)
if ck.get("hf_repo"):
    print("upload", hf_upload(path, ck["hf_repo"],
                              f"{ck['hf_path_in_repo']}/checkpoint_phase1.pt"))
print("To RESUME: re-run this notebook. Phase22Trainer restores weights, AdamW,")
print("GradScaler, RNG and the lossless byte-packer carry from local disk or the Hub.")

## Checkpoint / resume contract

`runs/phase22/checkpoints/checkpoint_phase1.pt` contains `schema=3` state:
fp32 master weights, AdamW moments, GradScaler, Python/Torch/CUDA RNG state, the
`BytePacker` carry (so no UTF-8 codepoint is split or lost across a restart) and
the dataset provenance manifest. `checkpoint_phase1_fp16.pt` is the 2-byte
inference-only copy. Set `HF_REPO` and re-run the notebook to resume after a
Colab disconnect.